In [1]:
from bs4 import BeautifulSoup
import requests
import re
import os
import json

In [2]:
amnt = 3

url = f"https://www.rottentomatoes.com/browse/movies_in_theaters/sort:popular?page={amnt}"
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

titles = []

for a in soup.select('a[data-track="scores"] span'):
    titles.append(a.text.strip())

titles = titles[:20000][::2]

In [3]:
normalized_titles = []

for t in titles:
    t = t.split(' |')[0]
    t = t.lower()
    t = t.replace('&', ' and ')
    t = re.sub(r"[^\w\s]", "", t)
    t = re.sub(r"\s+", "_", t.strip())
    normalized_titles.append(t)

In [4]:
qq = []
for i in normalized_titles:
    qq.append(f"https://www.rottentomatoes.com/m/{i}")

In [5]:
dataset = []

for url in qq:
    try:
        resp = requests.get(url, headers=headers, timeout=12)
        resp.raise_for_status()

        soup = BeautifulSoup(resp.text, "html.parser")
        ld_json = None
        script_ld = soup.find("script", type="application/ld+json")
        if script_ld:
            try:
                ld_json = json.loads(script_ld.string)
            except:
                pass

        canonical = soup.find("link", rel="canonical")
        clean_url = canonical["href"] if canonical else url
        director = "N/A"
        if ld_json and "director" in ld_json:
            dirs = ld_json["director"]
            if isinstance(dirs, list):
                director = ", ".join(d.get("name", "").strip() for d in dirs if d.get("name"))
            elif isinstance(dirs, dict):
                director = dirs.get("name", "N/A").strip()

        genre = "N/A"
        if ld_json and "genre" in ld_json:
            g = ld_json["genre"]
            genre = ", ".join(g) if isinstance(g, list) else str(g).strip()
        else:
            genre_tag = soup.find("rt-text", {"slot": "metadata-genre"})
            if genre_tag:
                genre = genre_tag.get_text(strip=True)

        orig_lang = "N/A"
        items = soup.find_all("div", class_="category-wrap")
        for item in items:
            label = item.find("rt-text", {"data-qa": "item-label"})
            if label and "Original Language" in label.get_text(strip=True):
                value = item.find("rt-text", {"data-qa": "item-value"})
                if value:
                    orig_lang = value.get_text(strip=True).strip()
                    break
        if orig_lang == "N/A":
            dt = soup.find(lambda tag: tag.name == "dt" and "Original Language" in tag.get_text(strip=True))
            if dt:
                dd = dt.find_next_sibling("dd")
                if dd:
                    orig_lang = dd.get_text(strip=True)

        year = "N/A"
        metadata_props = soup.find_all("rt-text", {"slot": "metadata-prop"})
        for prop in metadata_props:
            text = prop.get_text(strip=True)
            if re.match(r'^\d{4}$', text):  # просто 4 цифры — это год
                year = text
                break

        dataset.append({
            "url": clean_url,
            "director": director or "N/A",
            "genre": genre or "N/A",
            "original_language": orig_lang or "N/A",
            "release_year": year
        })

        print(f"{clean_url} Year: {year} | Director: {director} | Genre: {genre} | Lang: {orig_lang}")

    except Exception as e:
        print(f"Ошибка")

# Вывод датасета
for item in dataset:
    print(item)

#в CSV
import csv
with open("rotten_movies_full_dataset.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["url", "release_year", "director", "genre", "original_language"])
    writer.writeheader()
    writer.writerows(dataset)

print("V")

https://www.rottentomatoes.com/m/mercy Year: 2006 | Director: Patrick Roddy | Genre: Horror | Lang: English
https://www.rottentomatoes.com/m/return_to_silent_hill Year: N/A | Director: Christophe Gans | Genre: Horror, Mystery & Thriller | Lang: English
https://www.rottentomatoes.com/m/send_help Year: N/A | Director: Sam Raimi | Genre: Horror, Mystery & Thriller | Lang: English
https://www.rottentomatoes.com/m/one_battle_after_another Year: 2025 | Director: Paul Thomas Anderson | Genre: Mystery & Thriller, Comedy | Lang: English
https://www.rottentomatoes.com/m/28_years_later_the_bone_temple Year: N/A | Director: Nia DaCosta | Genre: Horror, Mystery & Thriller | Lang: English
https://www.rottentomatoes.com/m/sentimental_value Year: 2025 | Director: Joachim Trier | Genre: Drama | Lang: Norwegian
https://www.rottentomatoes.com/m/melania Year: N/A | Director: Brett Ratner | Genre: Documentary, Biography | Lang: English
https://www.rottentomatoes.com/m/the_housemaid Year: 2011 | Director: I